# AnyTime Universal Benchmark

Run early-exit benchmark for any AnyTime model. Switch by changing **one import line** below.

All HW metrics **per-PID** (process-scoped). Energy = `Î£ (power_w_i Ã— end_to_end_sec_i)` in Joules.

Per-run dir contains:
- `hw_results.json` â€” latency + memory + energy + per-PID GPU/CPU + FLOPs/params
- `quality_results.json` â€” accuracy / F1 / mAP / PPL (only if `skip_quality=False`)

Default sweep = HW-only. Pass `skip_quality=False` to enable quality eval.

Outputs:
- `logs/benchmark/{model}/...` â€” per-run JSONs (nested 2-3 levels deep)
- `results/{model}/` â€” CSV summaries


## 1. Setup

In [2]:
import numpy as np

In [3]:
import sys
import os
from pathlib import Path

REPO_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

repo root: D:\Research\earlyexit\EarlyExitMonoRepo


In [4]:
# HF login (only needed if pulling from a private repo)
from shared import load_env

load_env()

try:
    from google.colab import userdata

    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN", ""))
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF: logged in")
else:
    print("HF: no token, public repos only")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF: logged in


## 2. Choose model

**Change ONE import line below** to swap which model gets benchmarked.

In [5]:
# # === SWAP THIS LINE TO PICK MODEL ===
# # from benchmark_config import bert as cfg
# from benchmark_config import llama  as cfg
# # from benchmark_config import vision as cfg
# # from benchmark_config import yolo   as cfg

# print("model :", cfg.NAME)
# print("family:", cfg.MODEL_FAMILY)
# print("out   :", cfg.OUT_DIR)

## 3. Run full sweep

Default = HW-only (`skip_quality=True`). Each run does:
- `profile_hw()` â€” HW + latency + per-PID power/VRAM/CPU + FLOPs/params + EDP
- `evaluate_quality()` â€” (skipped by default)

Override sweep dims via `only_*` kwargs (see `cfg.run_all` signature).


In [6]:
# cfg.run_all(skip_quality=False)          # HW + quality (all datasets)
# # cfg.run_all()                            # HW-only (cnn_dailymail latency only)
# # cfg.run_all(skip_quality=False, only_exit=4)  # single exit smoke test

## 3b. Run ALL models (full sweep)

In [ ]:
# Run ALL models -- HW + quality sweep for every model family.
# Comment out any model you want to skip.
from benchmark_config import bert, llama, vision, yolo

for _cfg in [bert, llama, vision, yolo]:
    print(f"{60*chr(61)}")
    print(f"  {_cfg.NAME} ({_cfg.MODEL_FAMILY})")
    print(60*chr(61))
    try:
        _cfg.run_all(skip_quality=False)
    except Exception as _e:
        print(f"[{_cfg.NAME}] run_all failed: {_e}")
        import traceback; traceback.print_exc()


  bert (elasticbert-base)


In [ ]:
from shared import write_benchmark_csvs
from benchmark_config import bert, llama, vision, yolo
from collections import defaultdict
import pandas as pd

for _cfg in [bert, llama, vision, yolo]:
    out_dir = Path(_cfg.OUT_DIR)
    csv_root = REPO_ROOT / "results" / _cfg.NAME
    csv_root.mkdir(parents=True, exist_ok=True)

    hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
    q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
    run_dirs = list(hw_dirs | q_dirs)
    if not run_dirs:
        print(f"[{_cfg.NAME}] no results found, skipping")
        continue
    print(f"[{_cfg.NAME}] found {len(run_dirs)} run dirs ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

    groups = defaultdict(dict)
    for rd in run_dirs:
        rel_parent = rd.parent.relative_to(out_dir).as_posix()
        group_key = rel_parent.replace("/", "_") or "root"
        groups[group_key][rd.name] = rd

    for group_key, runs in sorted(groups.items()):
        csv_dir = csv_root / group_key
        csv_dir.mkdir(parents=True, exist_ok=True)
        write_benchmark_csvs(
            results_files=runs,
            out_dir=csv_dir,
            baseline_key=None,
            method_order=sorted(runs.keys()),
        )
        print(f"  {group_key}: {len(runs)} runs -> {csv_dir}")

print("All CSVs under:", REPO_ROOT / "results")


[bert] found 84 run dirs (84 hw, 84 quality)
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\CoLA\latency.csv
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\CoLA\energy.csv
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\CoLA\quality.csv
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\CoLA\hardware.csv
  CoLA: 12 runs -> D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\CoLA
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\MNLI\latency.csv
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\MNLI\energy.csv
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\MNLI\quality.csv
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\MNLI\hardware.csv
  MNLI: 12 runs -> D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\MNLI
[csv_export] wrote D:\Research\earlyexit\EarlyExitMonoRepo\results\bert\MRPC\latency.csv


## 4b. Export CSVs — ALL models

## 4. Export CSVs

Recursive walk over per-run dirs (handles 2-3 level nesting per model). Merges
`hw_results.json` + `quality_results.json` if both exist. Groups by parent dir.


In [ ]:
# from shared import write_benchmark_csvs
# from collections import defaultdict

# out_dir = Path(cfg.OUT_DIR)
# csv_root = REPO_ROOT / "results" / cfg.NAME
# csv_root.mkdir(parents=True, exist_ok=True)

# hw_dirs = {p.parent for p in out_dir.rglob("hw_results.json")}
# q_dirs = {p.parent for p in out_dir.rglob("quality_results.json")}
# run_dirs = list(hw_dirs | q_dirs)
# print(f"found {len(run_dirs)} run dirs under {out_dir} ({len(hw_dirs)} hw, {len(q_dirs)} quality)")

# groups = defaultdict(dict)
# for rd in run_dirs:
#     rel_parent = rd.parent.relative_to(out_dir).as_posix()
#     group_key = rel_parent.replace("/", "_") or "root"
#     groups[group_key][rd.name] = rd

# for group_key, runs in sorted(groups.items()):
#     csv_dir = csv_root / group_key
#     csv_dir.mkdir(parents=True, exist_ok=True)
#     write_benchmark_csvs(
#         results_files=runs,
#         out_dir=csv_dir,
#         baseline_key=None,
#         method_order=sorted(runs.keys()),
#     )
#     print(f"  wrote {len(runs)} runs -> {csv_dir}")

# print("CSVs under:", csv_root)


NameError: name 'cfg' is not defined

## 5. Quick view

In [ ]:
import pandas as pd

for csv_name in ['latency.csv', 'energy.csv', 'quality.csv', 'hardware.csv']:
    for f in csv_root.rglob(csv_name):
        rel = f.relative_to(csv_root)
        print(f'=== {rel} ===')
        print(pd.read_csv(f).head(8))
        print()


=== pretrained\latency.csv ===
    method  ttft_sec_mean  delta_ttft_pct  end_to_end_sec_mean  delta_e2e_pct  \
0   exit_0       0.006727             NaN             0.006727            NaN   
1   exit_1       0.007411             NaN             0.007411            NaN   
2  exit_10       0.022947             NaN             0.022947            NaN   
3  exit_11       0.023984             NaN             0.023984            NaN   
4  exit_12       0.025399             NaN             0.025399            NaN   
5  exit_13       0.027781             NaN             0.027781            NaN   
6  exit_14       0.028653             NaN             0.028653            NaN   
7  exit_15       0.030320             NaN             0.030320            NaN   

   per_sample_sec_mean  delta_per_sample_pct  throughput_samples_per_sec  
0             0.006727                   NaN                      77.468  
1             0.007411                   NaN                      75.185  
2             